### Setup

In [1]:
import pandas as pd
import os
import cv2
import ast
import tempfile
from typhoon_ocr import ocr_document
from bs4 import BeautifulSoup
import re
from rapidfuzz import fuzz, process
from tqdm import tqdm

In [2]:
os.environ["TYPHOON_OCR_API_KEY"] = "sk-ogvTcIwhoNXX69zmxalfZwZXW8JU5YcohnT4ISIgLTTapvJQ"

In [3]:
SUBMISSION_PATH = "../data/submission_template.csv"
TABLE_MAP_PATH = "../tableishere.csv"

In [4]:
template_old_df = pd.read_csv("../data/submission_template.csv")
template_new_df = pd.read_csv("../data/submission_template_v3.csv")
template_df = template_new_df.copy()
template_df['doc_id'] = template_old_df['doc_id']
template_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,0,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,0,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,0,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,0,party_list_34_11


In [5]:
# Load the table coordinate map
# filename (e.g. constituency_10_10_page2.png) -> coordinate [x1, y1, x2, y2]
table_map_df = pd.read_csv(TABLE_MAP_PATH)

# Parse coordinate strings like "[179, 503, 2388, 2470]" into actual lists
def parse_coord(val):
    try:
        return ast.literal_eval(val)
    except Exception:
        return None

table_map_df['coordinate'] = table_map_df['coordinate'].apply(parse_coord)

# Build a lookup dict: filename -> coordinate (None if no table detected)
table_coord_map = dict(zip(table_map_df['filename'], table_map_df['coordinate']))

table_map_df.head()

,filename,score,coordinate
0,constituency_10_1.png,NaN,None
1,constituency_10_10.png,NaN,None
2,constituency_10_10_page2.png,0.954209,"[179, 503, 2388, 2470]"
3,constituency_10_11.png,0.940642,"[389, 2772, 2313, 3095]"
4,constituency_10_11_page2.png,0.968331,"[399, 268, 2339, 3043]"


### EDA

In [6]:
template_df['party_name'].unique()

array(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
       'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
       'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
       'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
       'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
       'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
       'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
       'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
       'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', nan,
       'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
       'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
       'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
       'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
       'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
       'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธ

In [7]:
template_df.head()

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1


In [8]:
# invalid_name = ["Unknown Party", "พรรคที่ 1 (ไม่ระบุชื่อ)", "ไม่ระบุ"]

# NaN_df = template_df[
#     template_df['party_name'].isna() | template_df['party_name'].isin(invalid_name)
# ]

# NaN_df

In [9]:
# fix_invalid = {
#     2698: ('ไทยทรัพย์ทวี', 31),
#     2700: ('ใหม่', 149),
#     6461: ('ไทยทรัพย์ทวี', 473),
#     6892: ('ประชาชาติ', 632),
#     7031: ('ไทยทรัพย์ทวี', 2250),
#     7033: ('ใหม่', 481),
#     8000: ('ไทยทรัพย์ทวี', 420),
#     9426: ('ไทยทรัพย์ทวี', 347)
# }

In [10]:
# # Fix invalid
# for idx, (party, vote) in fix_invalid.items():
#     template_df.loc[idx, 'party_name'] = party
#     template_df.loc[idx, 'votes'] = vote

# for idx, (party, vote) in fix_invalid.items():
#     print(template_df.iloc[idx])
#     print()

### Extraction

In [11]:
def crop_table(image_path, coord):
    """Crop only the table region from an image using detected coordinates."""
    img = cv2.imread(image_path)
    x1, y1, x2, y2 = map(int, coord)
    cropped = img[y1:y2, x1:x2]
    return cropped

In [12]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [13]:
def extract_party_score_dict(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    table = soup.find("table")
    if table is None:
        raise ValueError("No <table> found")

    rows = table.find_all("tr")
    if not rows:
        return {}

    headers = [td.get_text(strip=True) for td in rows[0].find_all(["td", "th"])]

    # Candidate labels
    party_candidates = ["พรรคการเมือง", "ชื่อพรรคการเมือง", "สังกัดพรรคการเมือง"]
    score_candidates = ["ได้คะแนน"]

    party_idx = None
    score_idx = None
    best_party_score = 0
    best_score_score = 0

    for i, h in enumerate(headers):
        party_sim = max(fuzz.partial_ratio(h, c) for c in party_candidates)
        score_sim = max(fuzz.partial_ratio(h, c) for c in score_candidates)

        if party_sim > best_party_score:
            best_party_score = party_sim
            party_idx = i

        if score_sim > best_score_score:
            best_score_score = score_sim
            score_idx = i

    if best_party_score < 60 or best_score_score < 60:
        raise ValueError("Required columns not confidently found")

    result = {}

    for row in rows[1:]:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]

        if len(cols) <= max(party_idx, score_idx):
            continue

        key = cols[party_idx]
        value = cols[score_idx]

        try:
            value = thai_num_to_int(value)
        except Exception:
            continue

        result[key] = value

    return result

In [14]:
def extraction(path):
    """
    Run OCR only on images that have a detected table (score + coord in tableishere).
    If the filename is not in tableishere or has no score/coord, skip entirely.
    """
    try:
        if not os.path.exists(path):
            return {}

        filename = os.path.basename(path)
        coord = table_coord_map.get(filename)

        # Skip if no table was detected for this image (no score/coord in tableishere)
        if coord is None:
            return {}

        # Crop to table region and save to a temp file for OCR
        cropped = crop_table(path, coord)
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
            tmp_path = tmp.name
        cv2.imwrite(tmp_path, cropped)

        markdown = ocr_document(pdf_or_image_path=tmp_path)
        party_dict = extract_party_score_dict(markdown)

        if os.path.exists(tmp_path):
            os.remove(tmp_path)

        return party_dict

    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [15]:
def merge_pages(*pages):
    merged = {}
    for page in pages:
        if page is None:
            continue
        for k, v in page.items():
            merged[k] = merged.get(k, 0) + v
    return merged

In [16]:
def assign_votes(df, vote_dict, col='new_votes', threshold=80):
    """Assign extracted vote counts to `col` (default: new_votes)."""
    # Remove non-party keys
    vote_dict = {
        k: v for k, v in vote_dict.items()
        if k != 'รวมคะแนนทั้งสิ้น'
    }

    keys = list(vote_dict.keys())

    def get_vote(party_name):
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.ratio
        )

        if match is None:
            return 0

        best_key, score, _ = match

        if score >= threshold:
            return vote_dict[best_key]
        return 0

    df[col] = df['party_name'].apply(get_vote)
    return df

In [17]:
submission_df = template_df.copy()

# Initialize new_votes column
submission_df['new_votes'] = 0

In [18]:
PREFIX = "../data/images/"

doc_ids = submission_df['doc_id'].unique()

In [20]:
for doc_id in tqdm(doc_ids, desc="Processing", unit="doc"):
    try:
        pages = []

        for i in range(1, 11):
            if i == 1:
                path = PREFIX + doc_id + ".png"
            else:
                path = PREFIX + doc_id + f"_page{i}.png"

            pages.append(extraction(path))

        vote_dict = merge_pages(*pages)

        mask = submission_df['doc_id'] == doc_id

        submission_df.loc[mask] = assign_votes(
            submission_df.loc[mask].copy(),
            vote_dict,
            col='new_votes',
            threshold=80
        )

    except Exception as e:
        print(f"{doc_id}: {str(e)}")
    break

Processing:   0%|          | 0/300 [02:15<?, ?doc/s]

No <table> found


In [ ]:
submission_df

,id,party_name,votes,doc_id,new_votes
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1,14813
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1,14368
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1,979
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1,244
4,constituency_10_1_5,พลวัต,0,constituency_10_1,351
...,...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,0,party_list_34_11,0
10049,party_list_34_11_54,ความหวังใหม่,0,party_list_34_11,0
10050,party_list_34_11_55,ไทยรวมไทย,0,party_list_34_11,0
10051,party_list_34_11_56,เพื่อบ้านเมือง,0,party_list_34_11,0


In [ ]:
# Compare old votes vs new_votes
diff = submission_df[submission_df['votes'] != submission_df['new_votes']]
print(f"Rows with different values: {len(diff)}")
diff[['doc_id', 'party_name', 'votes', 'new_votes']].head(20)

In [ ]:
output_df = submission_df.drop(['doc_id', 'row_num', 'party_name'], axis=1)

# Use new_votes as the submission column
output_df = output_df.rename(columns={'new_votes': 'votes'}).drop(columns=['votes'], errors='ignore')

# Actually: keep new_votes as the primary votes for submission
output_df = submission_df[['votes', 'new_votes']].copy()
output_df.to_csv("submission_v2.csv", index=False)